<a href="https://colab.research.google.com/github/MoonlitTech/ddos-detection/blob/main/ddos_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP 365 Individual Project - DDoS Detection
DDoS Detection: Rule-Based Baseline

**Goal:**
Classify network flows as either benign or DDoS using a simple
threshold rule based on packets per second.

This is the baseline that will later be compared with machine-learning
models such as Logistic Regression, Decision Tree, and Random Forest.

EVALUATION MEASURES:
1. Precision: How often predicted DDoS attacks are actually DDoS.
2. Recall: How many actual DDoS attacks are correctly detected.
3. Inference time per flow: Average time needed to make a prediction.

This code uses only built-in Python libraries.

**AI disclosure: Document any AI assistance actually used in your
project in your README, following the assignment requirements.**

# 1. Import libraries and set up the dataset

* csv reads the dataset file.
* Math helps identify invalid numeric values such as infinity.
* Random shuffles the data before splitting it.
* Time measures how long predictions take.
* Set up the dataset and make the results repeatable

In [41]:
import csv
import math
import random
import time
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

CSV_PATH = "ddos-project/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
random.seed(42)
SEED = 42

# 2. Load and clean the data
Each flow has numeric features and a label: 1 = DDoS, 0 = not DDoS.

In [42]:
rows = []
feature_rows = []
seen_rows = set()
feature_names = []

with open(CSV_PATH, newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        row = {name.strip(): value for name, value in row.items()}

        # Skip exact duplicate rows.
        row_key = tuple(row.items())
        if row_key in seen_rows:
            continue
        seen_rows.add(row_key)

        # Skip rows without a usable packet rate.
        try:
            packet_rate = float(row["Flow Packets/s"])
        except (ValueError, TypeError):
            continue

        if not math.isfinite(packet_rate):
            continue

        label = 1 if row["Label"] == "DDoS" else 0
        rows.append((packet_rate, label))

        # Keep all numeric features for the machine-learning models.
        if not feature_names:
            feature_names = [name for name in row if name != "Label"]

        features = []
        for name in feature_names:
            try:
                value = float(row[name])
            except (ValueError, TypeError):
                value = float("nan")

            if not math.isfinite(value):
                value = float("nan")
            features.append(value)

        feature_rows.append(features)

print("Usable flows:", len(rows))

Usable flows: 223082


# 3. Split into training data and test data
Use the same flows for the baseline and all three machine-learning models.

In [43]:
order = list(range(len(rows)))
random.shuffle(order)
rows = [rows[i] for i in order]
feature_rows = [feature_rows[i] for i in order]

split_point = int(len(rows) * 0.7)
train = rows[:split_point]
test = rows[split_point:]
train_features = feature_rows[:split_point]
test_features = feature_rows[split_point:]

train_rates = [rate for rate, label in train]
train_labels = [label for rate, label in train]
test_rates = [rate for rate, label in test]
test_labels = [label for rate, label in test]

X_train = np.array(train_features)
X_test = np.array(test_features)
y_train = np.array(train_labels)
y_test = np.array(test_labels)

print("Training flows:", len(train))
print("Test flows:", len(test))

Training flows: 156157
Test flows: 66925


# 4. Create the rule-based baseline and its evaluation measures

In [44]:
def predict_rates(rates, threshold, direction):
    """Predict DDoS (1) or not DDoS (0) using the packet-rate threshold."""
    if direction == ">":
        return [1 if rate > threshold else 0 for rate in rates]
    return [1 if rate < threshold else 0 for rate in rates]


def get_precision_recall(labels, predictions):
    """Calculate precision and recall for DDoS predictions."""
    precision = precision_score(labels, predictions, zero_division=0)
    recall = recall_score(labels, predictions, zero_division=0)
    return precision, recall


def get_f1(precision, recall):
    """F1 is used internally to choose the baseline threshold."""
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# 5. Choose the threshold using training data only
Try thresholds from the training packet rates and both rule directions.

In [45]:
sorted_rates = sorted(train_rates)
best_f1 = -1
best_threshold = None
best_direction = None

for percent in range(1, 100):
    threshold = sorted_rates[int(len(sorted_rates) * percent / 100)]

    for direction in (">", "<"):
        predictions = predict_rates(train_rates, threshold, direction)
        precision, recall = get_precision_recall(train_labels, predictions)
        f1 = get_f1(precision, recall)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_direction = direction

print(
    "Baseline rule: predict DDoS if Flow Packets/s",
    best_direction,
    best_threshold,
)

Baseline rule: predict DDoS if Flow Packets/s < 2941.176471


# 6. Evaluate the baseline on the test data
* Apply the selected rule to the test data.
* Measure the time needed to make predictions.
* Divide total prediction time by the number of test flows
* to estimate the average inference time per flow.
* Precision and recall are calculated after timing, so the
* metric calculations do not inflate the prediction time.

In [46]:
start_time = time.perf_counter()
baseline_predictions = predict_rates(test_rates, best_threshold, best_direction)
baseline_elapsed = time.perf_counter() - start_time

baseline_precision, baseline_recall = get_precision_recall(
    test_labels, baseline_predictions
)
baseline_time = baseline_elapsed / len(test) if test else 0.0

# 7. Train the three machine-learning models
* Missing feature values are filled with the training-set median.
* Scaling is included in the same pipeline for each model.

In [47]:
models = {
    "Logistic Regression": Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2500, random_state=SEED)),
    ]),
    "Decision Tree": Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
        ("model", DecisionTreeClassifier(min_samples_leaf=5, random_state=SEED)),
    ]),
    "Random Forest": Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
        ("model", RandomForestClassifier(
            n_estimators=100,
            min_samples_leaf=2,
            random_state=SEED,
            n_jobs=-1,
        )),
    ]),
}

results = [{
    "Detector": "Threshold rule (baseline)",
    "Precision": baseline_precision,
    "Recall": baseline_recall,
    "Inference time (ms/flow)": baseline_time * 1000,
}]

for name, model in models.items():
    # Learn patterns from the training data.
    model.fit(X_train, y_train)

    # Warm up once, then time predictions on the same test data.
    model.predict(X_test[:min(1000, len(X_test))])
    start_time = time.perf_counter()
    predictions = model.predict(X_test)
    elapsed = time.perf_counter() - start_time

    precision, recall = get_precision_recall(y_test, predictions)
    time_per_flow = elapsed / len(X_test) if len(X_test) else 0.0

    results.append({
        "Detector": name,
        "Precision": precision,
        "Recall": recall,
        "Inference time (ms/flow)": time_per_flow * 1000,
    })

# 8. Display and save the comparison
Higher precision/recall are generally better; lower inference time is faster.

In [48]:
comparison = pd.DataFrame(results)
print("\nResults on the same held-out test data:")
print(comparison.round(6).to_string(index=False))

os.makedirs("results", exist_ok=True)
comparison.to_csv("results/metrics_table.csv", index=False)
print("\nSaved results to results/metrics_table.csv")


Results on the same held-out test data:
                 Detector  Precision   Recall  Inference time (ms/flow)
Threshold rule (baseline)   0.644922 0.998853                  0.000354
      Logistic Regression   0.998073 0.999010                  0.002408
            Decision Tree   0.999844 0.999713                  0.001685
            Random Forest   1.000000 0.999792                  0.012568

Saved results to results/metrics_table.csv
